# Exploratory Data Analysis

## Import dan Konfigurasi Library

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pd.set_option("display.max_rows", 60)

## 1. Exploratory Data Analysis pada Data Mentah

EDA pertama yang dilakukan merupakan EDA terhadap data mentah, yaitu sebelum dilakukannya drop kolom shortcut (ICULOS, Hour, HospAdmTim) maupun imputasi. 

In [ ]:
df = pd.read_csv("../data/Dataset.csv", index_col=0)
print("Shape:", df.shape)
df.head()

### 1.1 Analisis Missing Value (Sparsity)

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_table = missing_pct.to_frame("missing_pct").round(2)
print("Missing % per kolom (desc):")
print(missing_table.to_string())

focus = ["Lactate", "O2Sat", "pH", "PaCO2", "SaO2", "Resp"]
print("\nFocus clinical markers (missing %):")
print(missing_pct.loc[focus].round(2).to_string())

### 1.2 Asosiasi Fitur dengan Target Pearson r

Data null (NaN) diisi dengan median sebagai nilai sementara hanya untuk perhitungan korelasi (tidak di-persist ke dataframe utama).

In [ ]:
num = df.select_dtypes(include=[np.number]).copy()
num_filled = num.fillna(num.median(numeric_only=True))
corr = num_filled.corr(method="pearson")["SepsisLabel"].drop("SepsisLabel").abs()
top15 = corr.sort_values(ascending=False).head(15)
print("Top 15 Pearson r vs SepsisLabel:")
print(top15.round(4).to_string())

plt.figure(figsize=(8, 6))
sns.barplot(x=top15.values, y=top15.index, orient="h", color="steelblue")
plt.xlabel("Pearson r dengan SepsisLabel")
plt.title("Top 15 Fitur Terkorelasi dengan SepsisLabel")
plt.tight_layout()
plt.savefig("../reports/eda_feature_correlation.png", dpi=150)
plt.show()

### 1.3 Distribusi Kelas Target

In [ ]:
counts = df["SepsisLabel"].value_counts().sort_index()
pcts = (df["SepsisLabel"].value_counts(normalize=True).sort_index() * 100).round(3)
summary = pd.DataFrame({"count": counts, "pct": pcts})
print(summary)
print(f"\nImbalance ratio (neg:pos) = {counts.loc[0] / counts.loc[1]:.1f} : 1")

## 2.Balanced Resampling dan Secondary EDA

Langkah EDA ini terdiri dari pipeline preprocessing terkontrol untuk mencegah model mengambil shortcut administratif (ICULOS, Hour, HospAdmTime, Patient_ID) dan memastikan marker biologis (Lactate, O2Sat, dll.) mendominasi sinyal.

Langkah yang dilakukan untuk bagian ini antara lain:
1. Imputasi per pasien: groupby(Patient_ID).ffill() lalu fill sisa NaN dengan median global.
2. Downsampling kelas mayoritas: rasio 1:2 (negatif:positif) dengan random_state=42.
3. Secondary EDA: hitung ulang Pearson r pada dataset balanced.

### 2.1 Imputasi Timeline per-Pasien (ffill dan median)

In [ ]:
df_raw = pd.read_csv("../data/Dataset.csv", index_col=0)
df_raw = df_raw.sort_values(["Patient_ID", "Hour"]).reset_index(drop=True)
print("Raw shape:", df_raw.shape)
print("NaN total sebelum imputasi:", int(df_raw.isna().sum().sum()))

clinical_cols = [c for c in df_raw.columns if c not in ("Patient_ID", "SepsisLabel")]

df_imp = df_raw.copy()
df_imp[clinical_cols] = df_imp.groupby("Patient_ID")[clinical_cols].ffill()

global_medians = df_imp[clinical_cols].median(numeric_only=True)
df_imp[clinical_cols] = df_imp[clinical_cols].fillna(global_medians)

print("NaN total setelah ffill dan median:", int(df_imp.isna().sum().sum()))
df_imp.head()

### 2.2 Downsampling Kelas Mayoritas (rasio 1:2)

In [ ]:
pos = df_imp[df_imp["SepsisLabel"] == 1]
neg = df_imp[df_imp["SepsisLabel"] == 0]
print(f"Positif (Sepsis=1): {len(pos):,} rows")
print(f"Negatif (Sepsis=0): {len(neg):,} rows")

n_neg_sample = 2 * len(pos)
neg_sampled = neg.sample(n=n_neg_sample, random_state=42)

df_balanced = (
    pd.concat([pos, neg_sampled], axis=0)
      .sample(frac=1.0, random_state=42)
      .reset_index(drop=True)
)

print(f"\nBalanced dataset shape: {df_balanced.shape}")
print("Distribusi kelas (balanced):")
print(df_balanced["SepsisLabel"].value_counts().to_string())
print(f"\nRatio neg:pos = {(df_balanced['SepsisLabel']==0).sum() / (df_balanced['SepsisLabel']==1).sum():.2f} : 1")

### 2.3 Secondary EDA: Top 15 Pearson r pada Dataset Balanced

In [ ]:
admin_cols = ["ICULOS", "Hour", "HospAdmTime", "Patient_ID"]
df_eda = df_balanced.drop(columns=admin_cols)

num_b = df_eda.select_dtypes(include=[np.number])
corr_b = num_b.corr(method="pearson")["SepsisLabel"].drop("SepsisLabel").abs()
top15_b = corr_b.sort_values(ascending=False).head(15)

print("Top 15 Pearson r vs SepsisLabel (Balanced Dataset):")
print(top15_b.round(4).to_string())

plt.figure(figsize=(8, 6))
sns.barplot(x=top15_b.values, y=top15_b.index, orient="h", color="darkorange")
plt.xlabel("Pearson r dengan SepsisLabel")
plt.ylabel("Fitur")
plt.title("Top 15 Fitur Terkorelasi (Balanced Dataset)")
plt.tight_layout()
plt.savefig("../reports/eda_balanced_correlation.png", dpi=150)
plt.show()

In [ ]:
missing_pct_balanced = (df_balanced.isna().mean() * 100).sort_values(ascending=False)
missing_table_balanced = missing_pct_balanced.to_frame("missing_pct").round(2)
print("Missing % per kolom setelah imputasi (balanced dataset):")
print(missing_table_balanced.to_string())

focus = ["Lactate", "O2Sat", "pH", "PaCO2", "SaO2", "Resp"]
print("\nFocus clinical markers (missing %):")
print(missing_pct_balanced.loc[focus].round(2).to_string())

### Temuan EDA

1. Marker biologis utama seperti Lactate, O2Sat, pH, PaCO2, dan SaO2 memiliki tingkat missing yang sangat tinggi pada data mentah (93-97%). Hal ini menjustifikasi strategi imputasi per-pasien dengan ffill dan median global yang diterapkan di EDA bagian 2.
2. Kolom administratif seperti ICULOS dan HospAdmTime muncul di antara fitur terkorelasi tertinggi dengan SepsisLabel pada EDA awal. Ini merupakan kebocoran temporal (administrative leakage) yang harus dibuang sebelum training agar model tidak belajar dari shortcut non-biologis.
3. Ketidakseimbangan kelas yang ekstrem (rasio 54.6:1) menjustifikasi downsampling kelas mayoritas pada EDA bagian 2. Setelah balancing menjadi 1:2, ranking korelasi mencerminkan sinyal biologis yang lebih murni.

In [ ]:
df_balanced.to_csv("../data/df_balanced.csv", index=False)
print(f"Saved: ../data/df_balanced.csv | shape: {df_balanced.shape}")